In [1]:
import os
import json
import time
from openai import OpenAI

API_KEY = "sk-proj-xxxxxxxxxxxx" 
client = OpenAI(api_key=API_KEY)

In [2]:
# to avoid API failures

def generate_completion(model, system_prompt, user_content):
    
    """Helper to handle API calls with basic retry logic"""
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content}
                ],

                # for stable reasoning
                temperature=0.0
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2) # Wait 2s before retry
            else:
                print(f"  [!] Failed after {max_retries} attempts: {e}")
                return None

In [3]:
# API connection test
response = client.responses.create(
    model="gpt-4.1-mini",
    input="Say hello in one word"
)

print(response.output_text)

Hello!


In [4]:
# FUNSD (Safe Append Version)
INPUT_FILE = "phase1_funsd_train.jsonl"
OUTPUT_FILE = "phase1b_funsd_train_reasoning.jsonl"
MODEL = "gpt-4.1"

SYSTEM_PROMPT = """You are an expert Document AI data engineer. 
Your task is to write a brief, high-level semantic summary explaining the layout and content of a scanned form.

You will be provided with:
1. INPUT: The raw OCR text of the form (which contains text and bounding box coordinates).
2. TARGET JSON: The correct extracted data.

Your goal is to explain WHAT the document is and HOW the extracted keys and values logically relate to each other on the page.

CRITICAL RULES:
- Output ONLY a 'Thought:' block containing your summary. Do NOT output the JSON itself.
- NEVER use or mention bounding box coordinates (e.g., [120, 140, 200, 250]). Ignore all numbers in brackets entirely.
- Avoid mechanical spatial jargon (do not use phrases like "vertically aligned below", "x-coordinates", or "horizontal span").
- Focus purely on the semantic structure and meaning (e.g., "This document is a table detailing...", "The form contains a header section where the key X is paired with...").
- Keep it concise, readable, and descriptive.
"""

In [5]:
print(f"Starting FUNSD Processing ({INPUT_FILE})...")

if os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r') as f_in, open(OUTPUT_FILE, 'w') as f_out:
        lines = f_in.readlines()
        print(f"  -> Found {len(lines)} forms to process.")
        
        for i, line in enumerate(lines):
            data = json.loads(line)
            
            # preserve the Perfect Gold Answer
            existing_gold_answer = data['output']
            
            # ask LLM to explain gold asnwers too
            user_msg = f"Context: {data['input']}\n\nCorrect JSON: {existing_gold_answer}"
            
            # generate thought only
            thought_only = generate_completion(MODEL, SYSTEM_PROMPT, user_msg)
            
            if thought_only:
                # append 
                # If the LLM added "Thought:" prefix, keep it, otherwise add it.
                clean_thought = thought_only.replace("Thought:", "").strip()
                
                final_combined_output = f"Thought: {clean_thought}\nJSON: {existing_gold_answer}"
                
                # Update and Save
                data['output'] = final_combined_output
                f_out.write(json.dumps(data) + "\n")
            
            if (i+1) % 20 == 0: 
                print(f"  [FUNSD] Processed {i+1}/{len(lines)}")

    print(f"FUNSD Complete! Saved to {OUTPUT_FILE}")
else:
    print(f"Error: {INPUT_FILE} not found.")

Starting FUNSD Processing (phase1_funsd_train.jsonl)...
  -> Found 147 forms to process.
  [FUNSD] Processed 20/147
  [FUNSD] Processed 40/147
  [FUNSD] Processed 60/147
  [FUNSD] Processed 80/147
  [FUNSD] Processed 100/147
  [FUNSD] Processed 120/147
  [FUNSD] Processed 140/147
FUNSD Complete! Saved to phase1b_funsd_train_reasoning.jsonl
